---
title: Dataset Construction
date: 09/2025
authors:
  - name: James Butler
    affiliations: ucb
  - name: Michelle Maclennan
    affiliation: bas
  - name: Fernando Pérez
    affiliation: ucb
  - name: Jon McAuliffe
    affiliation: ucb
affiliations:
  - id: ucb
    institution: University of California Berkeley
    ror: https://ror.org/01an7q238
    department: Statistics
  - id: bas
    institution: British Antarctic Survey
    ror: https://ror.org/01rhff309
---

This notebook demonstrates some of our software workflows that can be used to construct tabular datasets of Antarctic AR events, including MERRA-2 streaming.

:::{attention}
If running the streaming workflows on CryoCloud, make sure to choose a server with enough RAM, as well as CPUs for parallelization. The RAM used by this notebook should not exceed 15GB at any point during the streaming loop, so any ser
:::

## Setup

First, we'll load up any packages and modules we might need.

In [1]:
# load external packages
import os
import pandas as pd
import xarray as xr
from pathlib import Path
import earthaccess
import ray
from tqdm import tqdm
from huggingface_hub import login, HfApi
import logging
import sys

In [2]:
# load internal helper modules
import artools
from artools.loading_utils import load_ais, load_cell_areas, EarthdataGatekeeper, load_catalog
from artools.display_utils import display_catalog
from artools.attribute_utils import *
from artools.compute_attributes_streaming import *

/home/jovyan/antarctic_AR_dataset/notebooks


In [3]:
login()

Next, let's load up the catalog. We provide a catalog of the first 250 storms in `subset_storms.h5`.

In [4]:
# load the catalog
storms = load_catalog('epsspace0.5_epstime12_minpts5_nreppts10_seed12345.h5')
# take only those that are landfalling
storms = storms[storms.is_landfalling]

Finally, we load up a mask of grid cells for Antarctica, as well as a mapping of grid cell to cell area.

In [5]:
ais_mask = load_ais()
cell_areas = load_cell_areas()

## Catalog Overview

Before we get started with the masking workflow, we can first take a quick look at the catalog. The catalog is a clustering of AR pixels identified in the [Wille (2021) catalog](https://doi.org/10.5281/zenodo.4009663), which takes an Eulerian approach to identifying atmoshperic rivers.

In [6]:
#display_catalog(storms, 5)

The `storms` object is a `pandas.DataFrame`, with rows corresponding to different identified AR storm events that made landfall. The `data_array` column contains binary-valued `xarray.DataArray` masks of the full spatiotemporal footprint of the storm. Note, we must use the `display_catalog` function from our local `display_utils` module to view the dataframe. Otherwise, if one attempts to print the dataframe as usual, the `data_array` column will be filled with string representations of the masks which (1) take forever to render and (2) are unpleasant to look at.

In [7]:
#storms.loc[1].data_array

Organizing the catalog in this way means we can appeal to `pandas` very powerful and compact API to compute relevant storm quantities.

## Filling in Storm Attributes and Impacts

Now let's put these masks to good use and start grabbing interesting quantities associated with each storm. There are two kinds of quantities we can compute, ones which need reanalysis data (geographic attributes, when and where, etc.) and ones which don't (atmospheric quantities). We'll start with computing quantities that don't yet require reanalysis data and finish with a fully cloud-based workflow to compute quantities that do require reanalysis.

### Geographic Quantities

Let's start with computing geographic attributes of each landfalling storm. This includes the following quantities:
+ `max_area`: the largest areal extent of the storm over its complete lifetime
+ `mean_landfalling_area`: the average landfalling areal extent of the storm
+ `cumulative_landfalling_area`: the cumulative time x space the storm spent over the ice sheet
+ `duration`: how long did it last
+ `start_date`: when did the storm first appear
+ `end_date`: when did it dissipate
+ `max_south_extent`: how far south did it go, in degrees latitude?
+ `region`: the region of Antarctica it made landfall in (either `West`, `East 1`, or `East 2`)

Helper functions to compute each of these attributes given a storm's `xarray.DataArray` can be found in the `attributes_utils` module.

In [8]:
storms['max_area'] = storms['data_array'].apply(lambda x: compute_max_area(x, cell_areas)).astype(int)
storms['mean_landfalling_area'] = storms['data_array'].apply(lambda x: compute_mean_area(x, cell_areas, ais_mask)).astype(int)
storms['cumulative_landfalling_area'] = storms['data_array'].apply(lambda x: compute_cumulative_spacetime(x, cell_areas, ais_mask)).astype(int)
storms['duration'] = storms['data_array'].apply(compute_duration)
storms['start_date'] = storms['data_array'].apply(add_start_date)
storms['end_date'] = storms['data_array'].apply(add_end_date)
storms['max_south_extent'] = storms['data_array'].apply(compute_max_southward_extent)

In [9]:
# regions defined by ranges of longitudes
region_defs = {'West': [-150, -30], 
               'East 1': [-30, 75],
               'East 2': [75, -150]}
region_masks = find_region_masks(region_defs, ais_mask)
storms['region'] = storms['data_array'].apply(lambda x: find_landfalling_region(x, cell_areas, region_masks))

Let's take a look at what we were able to create.

In [10]:
#display_catalog(storms, 5)

### Atmospheric Quantities

Now for the more interesting part, let's compute atmospheric characteristics and attributes for each storm. This includes things like their landfalling moisture content, wind speeds, as well as their impacts, like cumulative snowfall on the ice sheet or maximum surface temperature anomaly.

We'll be using MERRA-2 reanalysis data to get these quantities. Of course, one might just execute this workflow by loading it up from disk at some local file system you have accxess to, but that's not very reproducible, accessible, or open! Instead, we'll be streaming MERRA-2 from AWS S3 buckets using `earthaccess`, a workflow that is much more reproducible, accessible, open, and lightweight!

:::{attention}

To access MERRA-2 data in the cloud, you'll need to first create a NASA Earthdata account and create the relevant [Earthdata prerequisite files](https://github.com/nasa/gesdisc-tutorials/blob/main/notebooks/How_to_Generate_Earthdata_Prerequisite_Files.ipynb), most notably the `.netrc` file with your Earthdata login credentials. You will also need to be in the AWS `us-west-2` region to stream from s3. If you are running this notebook on CryoCloud, you are in `us-west-2`.

:::

For this tutorial, we'll just be getting all of the variables from one MERRA-2 dataset. In our attempts with multiple datasets, we have found difficulties with the parallelization (hanging processes, etc.) So, we'll be demonstrating on a single dataset for now while we work out the kinks for using other datasets.

In [11]:
data_dois = {'climatology': '10.5067/5ESKGQTZG7FO',
             'T2M_TQV_SLP': '10.5067/3Z173KIE2TPD',
             'VFLXQV_PRECIP': '10.5067/Q5GVUVUIVGO7',
             'V850': '10.5067/VJAFPLI1CSIV',
             'OMEGA': '10.5067/QBZ6MG944HW0'}

#### Climatology

A couple of the characteristics we want to compute are anomalies, such as the maximum landfalling 2m temperature anomaly underneath the footprint of the AR, as well as integrated water vapor anomaly. We'll compute the climatology of both of these variables via streaming first. The climatology is computed by taking monthly averaged dataset of our variables from MERRA-2, and then further averaging those across the years. Since the original Wille (2021) catalog on which ours is based goes from 1980 to 2022, we will be computing monthly averages over this range.

:::{attention}
Either we can run the next cell to create the climatology dataset, which should take about 20 minutes to do. Or, you could run this just once, save it, and just load it up from disk for future runs. If you don't want to wait, we've already stored it in the `output` folder, so you can skip this cell and run the next one.
:::

In [12]:
# grab granules for monthly data
#granule_lst = earthaccess.search_data(doi=data_dois['climatology'], 
#                                  temporal=('1980-01-01', '2022-12-31'))
# open pointers to nc4 stored in AWS S3 buckets
#pointers = earthaccess.open(granule_lst, show_progress=False)

#monthly_means = xr.open_mfdataset(pointers)
#monthly_means = monthly_means[['T2M', 'TQV']]
#climatology_ds = monthly_means.groupby(monthly_means.time.dt.month).mean().compute()
#climatology_ds = climatology_ds.assign_coords(lat=climatology_ds.lat.round(5), lon=climatology_ds.lon.round(5))
#climatology_ds.to_netcdf('output/climatology.nc4')

In [13]:
# or run this!
climatology_ds = xr.load_dataset('../output/data_products/climatology.nc4')

#### Ray Setup

We will be streaming quantities from a MERRA-2 dataset and extracting summary statistics on a storm-by-storm basis. To save on time, we opt to process the storms in parallel using Ray. Here, we set up our Ray instance, using 4 workers to execute the parallel processing.

In [14]:
sorted_storms = storms.sort_values(by='duration', ascending=False)

We will want to log the progress of our computation. Let's set up a logger here.

In [15]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('background_processing.log')
    ]
)
logging.info("Starting notebook background execution...")

2026-03-13 22:35:34,478 - INFO - Starting notebook background execution...


#### Integrated Water Vapor, 2m-Temperatures, Sea-Level Pressure

This includes IWV, IWV anomaly, 2m-temperature, 2m-temperature anomaly, and sea level pressure quantities.

In our workflow, we go through each storm and execute a function called `compute_summaries` on each storm. `compute_summaries` computes scalar summaries of all desired variables from the particular MERRA-2 dataset we are
working with. To indicate which variables and which scalar summaries you would like, you must pass in a dictionary of dictionaries which we will call `func_vars_dict`. The keys of the outer dictionary are the variable names in the MERRA-2 dataset we'd like to compute aggregates of, and the values are themselves dictionaries where each item represents some desired aggregation function of that variable. The provided keys are the names of that scalar summary that will appear in the resulting dataset.

In this way, when we stream a set of days of data for a particular storm, we compute all relevant quantities on that storm before throwing the data away and re-streaming for the next storm.

In [16]:
num_cpus = 10

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

/srv/conda/envs/notebook/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


We also sort the storms by their duration in descending order. The longer a storm lasts, the longer it takes to stream its data. So, sorting ensures our workers are each working on storms of similar lengths and thus avoid any overhead with some tasks finishing much more quickly than others.

In [ ]:
func_vars_dict = {'T2M': {'max_T2M_anomaly_ais': lambda storm_da, var_da, area_da: 
                              compute_max_intensity(storm_da, var_da, area_da, ais_mask)},
                  'TQV': {'max_IWV_ais': lambda storm_da, var_da, area_da: 
                              compute_max_intensity(storm_da, var_da, area_da, ais_mask)},
                  'SLP': {'max_ocean_SLP_gradient': lambda storm_da, var_da, area_da: 
                              compute_max_SLPgrad(storm_da, var_da, area_da, ais_mask)}}

func_vars_dict_ref = ray.put(func_vars_dict)

data_doi = data_dois['T2M_TQV_SLP']
# list with the computed results
results_T2M_TQV_SLP = []

chunk_size = 10
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = 10 # cap the number of in-flight Ray tasks

logging.info(f"Starting T2M_TQV_SLP computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # submit ONLY this batch of tasks to Ray
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            climatology_ds=climatology_ref
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results and immediately clean up memory
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_T2M_TQV_SLP.extend(results)
        
        # Clean up the reference immediately so Ray empties the Object Store
        batch_refs[j] = None
        
    logging.info(f"T2M_TQV_SLP: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("T2M_TQV_SLP computations complete.")

2026-03-13 22:35:37,889 - INFO - Starting T2M_TQV_SLP computations in batches...


In [ ]:
# construct dataframe to concatenate into the original dataframe
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_T2M_TQV_SLP = pd.DataFrame(results_T2M_TQV_SLP, columns=labels, index=sorted_storms.index)
df_T2M_TQV_SLP.to_csv('../output/results_T2M_TQV_SLP.csv')
logging.info("T2M_TQV_SLP results saved!")

In [ ]:
ray.shutdown()

#### Poleward Integrated Vapor Transport

Next, we compute poleward integrated vapor transport quantities, or vIVT quantities. The computation is very similar to the above, using the same aggregation functions, passed into the overall `compute_summaries` function. However, this time we aren't computing any anomalies so no need to pass in any climatologies, and we set the `half_hour` flag to true, since this dataset in MERRA-2 is hourly on the half-hour..

First, we reinitialize the cluster.

In [ ]:
num_cpus = 10

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'VFLXQV': {'max_vIVT_ais': lambda storm_da, var_da, area_da: 
                                 compute_max_intensity(storm_da, -var_da, area_da, ais_mask)}}

data_doi = data_dois['VFLXQV_PRECIP']
func_vars_dict_ref = ray.put(func_vars_dict)

In [ ]:
results_VFLXQV = []

chunk_size = 10
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = 10

logging.info(f"Starting VFLXQV computations in batches...")

# iterate through the data in chunks of 15
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    
    # submit ONLY this batch of tasks to Ray
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            half_hour=True
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_VFLXQV.extend(results)
        
        # clean up the reference immediately
        batch_refs[j] = None 
        
    logging.info(f"VFLXQV: Finished up to chunk {min(i+batch_size, len(chunks))}/{len(chunks)}")

logging.info("VFLXQV computations complete.")

In [ ]:
# construct dataframe to concatenate into the original dataframe
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_VFLXQV = pd.DataFrame(results_VFLXQV, columns=labels, index=sorted_storms.index)
df_VFLXQV.to_csv('../output/results_VFLXQV.csv')
logging.info("VFLXQV results saved!")

In [ ]:
ray.shutdown()

#### Snowfall and Rainfall

Snowfall and rainfall come from the same MERRA-2 dataset as poleward integrated vapor transport. However, we treat them separately because precipitation quantities are computed not on the spatiotemporal footprint of the mask as the previous quantities, but an augmented mask where even if the AR has left a pixel at a particular time, that pixel is still considered part of the AR footprint for the next 24 hours. Because of this extra preprocessing, we pass a `precip_func` function, the aggregation function we wish to use, into a summary computation function particular to this case (`compute_precip_summaries`).

In [ ]:
num_cpus = 10

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
data_doi = data_dois['VFLXQV_PRECIP']

# storing the aggregation function for precip in the ray store
def precip_func(storm_da, var_da, area_da):
    return compute_cumulative(storm_da, var_da, area_da, ais_mask)
precip_func_ref = ray.put(precip_func)

In [ ]:
results_precip = []

chunk_size = 10
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = 10 # cap the number of in-flight Ray tasks

logging.info(f"Starting Precipitation computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # submit ONLY this batch of tasks to Ray
    batch_refs = [
        compute_precip_chunk_summaries.remote(
            chunk, 
            cell_areas_ref, 
            precip_func_ref,
            data_doi,
            gatekeeper=gatekeeper
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results and immediately clean up memory
    for j, ref in enumerate(batch_refs):
        # Retrieve the result from the Ray Object Store
        results = ray.get(ref)

        results_precip.extend(results)
        
        # Clean up the reference immediately so Ray empties the RAM
        batch_refs[j] = None
        
    logging.info(f"Precipitation: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("Precipitation computations complete.")

In [ ]:
# construct dataframe to concatenate into the original dataframe
labels = ['cumulative_rainfall_ais', 'cumulative_snowfall_ais']
df_precip = pd.DataFrame(results_precip, columns=labels, index=sorted_storms.index)
df_precip.to_csv('../output/results_precip.csv')
logging.info("Precip results saved!")

In [ ]:
ray.shutdown()

#### 850 hPa Poleward Wind

Now, we compute aggregates of the 850 hPa poleward wind, at the time of first landfall, under the portion of the AR footprint which is still over the ocean. This partially because 850 hPa wind is null on the continent, and this gives us some notion of how windy conditions were at landfall. To account for this, we use an alternative aggregation function `compute_max_landfalling_wind`.

In [ ]:
num_cpus = 10

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'V850': {'max_landfalling_v850hPa': lambda storm_da, var_da, area_da: 
                           compute_max_landfalling_wind(storm_da, -var_da, area_da, ais_mask)}}
data_doi = data_dois['V850']
results_V850 = []

func_vars_dict_ref = ray.put(func_vars_dict)

chunk_size = 10
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = 10

logging.info(f"Starting V850 Wind computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # submit ONLY this batch of tasks to Ray
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper,
            half_hour=True
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results and immediately clean up memory
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_V850.extend(results)
        
        # clean up the reference immediately so Ray empties the RAM
        batch_refs[j] = None
        
    logging.info(f"V850 Wind: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("V850 Wind computations complete.")

In [ ]:
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_wind = pd.DataFrame(results_V850, columns=labels, index=sorted_storms.index)
df_wind.to_csv('../output/results_V850.csv')
logging.info("V850 wind results saved!")

In [ ]:
ray.shutdown()

#### Omega

Finally, we compute omega, or some notion of lifting in the storm. This is computed by first finding the portion of the AR footprint over the ice sheet at the time of first landfall. At these points, we find the minimum omega over all atmospheric levels, and then aggregate using a spatial average over that landfalling portion of the AR. To compute this, like for the last quantity, we use a particular aggregation function called `compute_avg_landfalling_minomega`.

In [ ]:
num_cpus = 10

ray.init(num_cpus=num_cpus, logging_level='ERROR', 
         _metrics_export_port=-1, include_dashboard=False, 
         log_to_driver=False, runtime_env={'py_modules': [artools.attribute_utils, artools.loading_utils]})

# put the climatologies and cell areas in the ray object store
climatology_ref = ray.put(climatology_ds)
cell_areas_ref = ray.put(cell_areas)
# prevents parallel open requests from being made to NASA servers
gatekeeper = EarthdataGatekeeper.remote()

In [ ]:
func_vars_dict = {'OMEGA': {'avg_landfalling_minomega': lambda storm_da, var_da, area_da: 
                             compute_avg_landfalling_minomega(storm_da, var_da, area_da, ais_mask)}}
data_doi = data_dois['OMEGA']
results_omega = []

func_vars_dict_ref = ray.put(func_vars_dict)

chunk_size = 10
chunks = [sorted_storms[i:i + chunk_size] for i in range(0, sorted_storms.shape[0], chunk_size)]

batch_size = 10 # cap the number of in-flight Ray tasks

logging.info(f"Starting OMEGA computations in batches...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i + batch_size]
    
    # submit ONLY this batch of tasks to Ray
    batch_refs = [
        compute_chunk_summaries.remote(
            chunk,
            func_vars_dict_ref,
            cell_areas_ref,
            data_doi,
            gatekeeper=gatekeeper
        ) for chunk in batch_chunks
    ]
    
    # gather this batch's results and immediately clean up memory
    for j, ref in enumerate(batch_refs):
        results = ray.get(ref)
        results_omega.extend(results)
        
        # clean up the reference immediately so Ray empties the RAM
        batch_refs[j] = None
        
    logging.info(f"OMEGA: Finished up to chunk {min(i + batch_size, len(chunks))}/{len(chunks)}")

logging.info("OMEGA computations complete.")

In [ ]:
labels = [key for quantity_dict in func_vars_dict.values() for key in quantity_dict.keys()]
df_omega = pd.DataFrame(results_omega, columns=labels, index=sorted_storms.index)
df_omega.to_csv('../output/results_omega.csv')
logging.info("OMEGA wind results saved!")

In [ ]:
ray.shutdown()